# Zepto Data Pipeline: Catalog Scraping to Normalized Relational Store
This notebook demonstrates an end-to-end raw-to-relational ETL pipeline:
1. Scrape catalog items from books.toscrape.com across multiple categories.
2. Clean and parse types (floats, integers, booleans).
3. Apply fixed currency conversion: 1 GBP = 105.50 INR.
4. Store in a normalized 2-table SQLite schema (categories, books).
5. Execute 5 analytical SQL queries.
6. Verify equivalence between pd.read_sql and pd.merge.


In [1]:
import os
import re
import sqlite3
import requests
from bs4 import BeautifulSoup
import pandas as pd

from pipeline import clean_data, scrape_category, init_db, load_to_sqlite, run_queries, verify_pandas_equivalence, CATEGORIES_TO_SCRAPE, DB_PATH

# Scrape catalog data
all_books = []
for cat_name, rel_url in CATEGORIES_TO_SCRAPE:
    print(f"Scraping category: {cat_name}...")
    cat_books = scrape_category(cat_name, rel_url)
    print(f"  Fetched {len(cat_books)} items.")
    all_books.extend(cat_books)

# Clean data
cleaned_df = clean_data(all_books)
print(cleaned_df.head())


In [2]:
# Initialize DB and Load
conn = init_db(DB_PATH)
load_to_sqlite(cleaned_df, conn)

# Execute 5 SQL queries
query_results = run_queries(conn)


In [3]:
# Verify Equivalence between pd.read_sql and pd.merge
df_sql, df_pandas = verify_pandas_equivalence(conn)
conn.close()
